In [ ]:
# Extracting "Case Facts" into a Persistent Block
# System prompt includes a structured facts block
# that is updated after each turn but never summarized:

<case_facts>
  customer_name: Jane Doe
  account_id: ACC-2024-55912
  issue_type: billing_dispute
  disputed_amount: $147.50
  disputed_date: 2024-03-01
  order_id: ORD-2024-78341
  resolution_offered: full_refund
  resolution_status: pending_approval
  escalation_history: none
  turns_elapsed: 12
</case_facts>

# Conversation history is summarized, but case_facts
# block is preserved verbatim across all turns.

# Subagent Output Format
{
  "findings": [
    {
      "claim": "Authentication uses JWT with 24h expiry",
      "source_file": "src/auth/jwt_handler.py",
      "source_line": 42,
      "confidence": "high"
    },
    {
      "claim": "Rate limiting is 100 req/min per API key",
      "source_file": "src/middleware/rate_limit.py",
      "source_line": 18,
      "confidence": "high"
    }
  ],
  "files_examined": 23,
  "gaps": ["Could not determine session storage backend"]
}

# Example TODO

# Structured Error Context
When a tool call fails, capture structured information about the failure rather than just "an error occurred." This enables downstream processing to make informed decisions:
{
  "error_type": "database_timeout",
  "attempted_action": "fetch customer order history",
  "partial_results": "Retrieved 3 of estimated 12 orders before timeout",
  "alternatives": [
    "Retry with narrower date range",
    "Use cached results from last successful query (2 hours old)",
    "Proceed with partial results and note the gap"
  ],
  "impact_on_response": "Cannot provide complete order history"
}

# Example TODO

# 5. Codebase Exploration & Confidence

# Scratchpad Files for Persisting Findings
A practical mitigation for context degradation: write findings to a scratchpad file as you discover them. This creates a persistent record that survives context window turnover:

# .claude/scratchpad.md
# Written by Claude during exploration, read back when needed

## Authentication Flow
- Entry point: src/auth/router.py:login() (line 45)
- Calls: src/auth/service.py:authenticate(email, password)
- JWT creation: src/auth/jwt.py:create_token() with 24h expiry
- Token stored: HttpOnly cookie named "session_token"
- Refresh: src/auth/router.py:refresh() (line 78)

## Database Access
- ORM: SQLAlchemy with async sessions
- Session factory: src/db/session.py:get_session()
- Migrations: Alembic in migrations/ directory
- Models: src/models/ (User, Order, Product, Payment)

## Open Questions
- [ ] Where is rate limiting configured?
- [ ] How are background jobs scheduled?

# Example to write scratchpad findings in code: TODO




